In [ ]:
from pathlib import Path
import polars as pl
import plotly.express as px
import numpy as np
import polars as pl
import scipy.stats as stats
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Resolver la raíz del proyecto estando dentro de /notebooks/
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_CSV_PATH = BASE_DIR / "data" / "raw" / "Lluvia_2026_v2.csv"

print(f"Ruta apuntada: {RAW_CSV_PATH}")
print(f"¿Existe el archivo?: {RAW_CSV_PATH.exists()}")

In [ ]:
df_raw = pl.read_csv(
    RAW_CSV_PATH,
    separator=";",
    schema_overrides={
        "hora": pl.Int32,
        "fecha": pl.Float64,
        "lluvia_mm": pl.Float64,
    },
)

# Salida ad hoc: inspección de estructura inicial
df_raw.glimpse()

In [ ]:
df_base = (
    df_raw.with_columns(
        fecha_date=pl.date(1899, 12, 30) + pl.duration(days=pl.col("fecha"))
    )
    .with_columns(
        datetime=pl.col("fecha_date").dt.combine(
            pl.time(hour=pl.col("hora"),
                    minute=0, second=0)
        )
    )
    .sort("datetime")
)

# Salida ad hoc: verificar la conversión temporal
df_base.select(["fecha", "fecha_date", "hora", "datetime", "lluvia_mm"]).head(5)

In [ ]:
target_hours = [6] + list(range(12, 97, 12))

rolling_exprs = [
    pl.col("lluvia_mm")
    .rolling_mean(window_size=(h // 3),
                  min_samples=(h // 3))
    .alias(f"ma_{h}h")
    for h in target_hours
]

df_matrix = df_base.with_columns(
    [pl.col("lluvia_mm").cum_sum().alias("sum_acum")] + rolling_exprs
)

# Salida ad hoc: revisar la matriz resultante
df_matrix.head(10)

In [16]:
ma_cols = [f"ma_{h}h" for h in target_hours]

df_plot = df_matrix.with_columns(
    pl.col("datetime").dt.strftime("%Y-%m-%d %H:00").alias("datetime_str")
)

timestamps = df_plot["datetime_str"].to_list()
matrix_values = df_plot.select(ma_cols).to_numpy().T

fig = px.imshow(
    matrix_values,
    labels=dict(x="Tiempo",
                y="Ventana Temporal",
                color="Precipitación Prom. (mm)"),
    x=timestamps,
    y=[f"{h}h" for h in target_hours],
    color_continuous_scale="Reds",
    title="Mapa de Calor: Acumulación Temporal de Lluvias (mm) por Ventana Móvil y Variable",
    aspect="auto",
)

fig.update_xaxes(side="bottom",
                 tickangle=-45)
fig.show()

In [19]:
display(df_matrix)
display(df_matrix.describe())

hora,fecha,lluvia_mm,fecha_date,datetime,sum_acum,ma_6h,ma_12h,ma_24h,ma_36h,ma_48h,ma_60h,ma_72h,ma_84h,ma_96h
i32,f64,f64,date,datetime[μs],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2,46219.0,0.2,2026-07-16,2026-07-16 02:00:00,0.2,null,null,null,null,null,null,null,null,null
5,46219.0,0.0,2026-07-16,2026-07-16 05:00:00,0.2,0.1,null,null,null,null,null,null,null,null
8,46219.0,0.8,2026-07-16,2026-07-16 08:00:00,1.0,0.4,null,null,null,null,null,null,null,null
11,46219.0,3.1,2026-07-16,2026-07-16 11:00:00,4.1,1.95,1.025,null,null,null,null,null,null,null
14,46219.0,6.0,2026-07-16,2026-07-16 14:00:00,10.1,4.55,2.475,null,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
17,46223.0,0.0,2026-07-20,2026-07-20 17:00:00,264.6,4.0,3.15,3.825,5.066667,4.73125,4.175,4.020833,3.610714,3.19375
20,46223.0,3.1,2026-07-20,2026-07-20 20:00:00,267.7,1.55,2.775,3.2125,4.408333,4.7375,4.33,4.15,3.692857,3.28125
20,46223.0,0.0,2026-07-20,2026-07-20 20:00:00,267.7,1.55,2.775,3.2125,4.058333,4.4375,4.13,3.941667,3.65,3.275


statistic,hora,fecha,lluvia_mm,fecha_date,datetime,sum_acum,ma_6h,ma_12h,ma_24h,ma_36h,ma_48h,ma_60h,ma_72h,ma_84h,ma_96h
str,f64,f64,f64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",56.0,56.0,56.0,"""56""","""56""",56.0,55.0,53.0,49.0,45.0,41.0,37.0,33.0,29.0,25.0
"""null_count""",0.0,0.0,0.0,"""0""","""0""",0.0,1.0,3.0,7.0,11.0,15.0,19.0,23.0,27.0,31.0
"""mean""",12.5,46221.428571,4.816071,"""2026-07-18 10:17:08.571428""","""2026-07-18 22:47:08.571428""",163.919643,4.901818,5.063208,5.215306,5.083333,4.835976,4.512703,4.329293,4.267734,4.308
"""std""",6.936072,1.412376,5.192189,null,null,78.832025,4.495615,4.185723,3.700942,3.242928,2.631324,1.965145,1.467253,1.166023,1.048291
"""min""",2.0,46219.0,0.0,"""2026-07-16""","""2026-07-16 02:00:00""",0.2,0.1,0.275,0.4625,0.833333,1.4375,1.69,2.320833,2.925,3.015625
"""25%""",8.0,46220.0,0.2,"""2026-07-17""","""2026-07-17 20:00:00""",138.1,1.55,1.9,2.45,2.6,3.0,3.5,3.304167,3.353571,3.28125
"""50%""",14.0,46222.0,3.1,"""2026-07-19""","""2026-07-19 08:00:00""",168.1,3.9,3.575,4.2125,4.433333,4.43125,4.175,3.941667,3.703571,4.184375
"""75%""",17.0,46223.0,10.0,"""2026-07-20""","""2026-07-20 02:00:00""",219.0,7.6,7.55,6.8875,6.283333,5.525,5.05,5.25,5.310714,5.51875
"""max""",23.0,46223.0,18.0,"""2026-07-20""","""2026-07-20 23:00:00""",269.7,17.5,14.25,13.125,12.333333,10.05,8.135,6.8875,6.321429,5.778125


In [ ]:
# Celda 6: Exportación de matriz completa a Parquet
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True,
                    exist_ok=True)

OUTPUT_PARQUET_PATH = PROCESSED_DIR / "lluvia_2026_matrix.parquet"

# Exportar preservando nulos (ventanas de inicialización) y tipos de datos
df_matrix.write_parquet(OUTPUT_PARQUET_PATH)

In [ ]:
# 1. Cargar matriz procesada desde el Parquet exportado
parquet_path = OUTPUT_PARQUET_PATH
df_parquet = pl.read_parquet(parquet_path)

# Definir las series temporales de interés
hours = [6] + list(range(12, 97, 12))
ma_cols = [f"ma_{h}h" for h in hours]

# Distribuciones candidatas para modelar precipitación y acumulados
candidate_dists = {
    "Gumbel": stats.gumbel_r,
    "Gamma": stats.gamma,
    "Log-Normal": stats.lognorm,
    "Weibull": stats.weibull_min,
    "Normal": stats.norm
}

# Configuración del Grid para subplots (3 filas x 3 columnas)
fig = make_subplots(
    rows=3, cols=3,
    subplot_titles=[f"Serie Ventana: {h}h" for h in hours],
    vertical_spacing=0.1,
    horizontal_spacing=0.08
)

best_fits_summary = {}

# 2. Iterar sobre cada serie de tiempo para ajuste y evaluación
for idx, h in enumerate(hours):
    col = f"ma_{h}h"
    
    # Extraer datos, descartando el warm-up (nulos) y ceros (períodos secos)
    data_raw = df_parquet.select(col).drop_nulls()[col].to_numpy()
    data = data_raw[data_raw > 0]
    
    if len(data) < 10:
        continue

    # Calcular histograma empírico de la acumulación real
    counts, bin_edges = np.histogram(data, bins=15, density=True)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    
    row = (idx // 3) + 1
    col_idx = (idx % 3) + 1
    
    # Graficar datos empíricos
    fig.add_trace(
        go.Bar(
            x=bin_centers, 
            y=counts, 
            name="Empírico", 
            marker_color="rgba(200, 200, 200, 0.6)",
            showlegend=(idx == 0)
        ),
        row=row, col=col_idx
    )
    
    x_plot = np.linspace(min(data), max(data), 100)
    best_stat = float("inf")
    best_dist_name = ""
    
    # Evaluar cada distribución candidata
    for name, dist in candidate_dists.items():
        try:
            params = dist.fit(data)
            
            # Prueba de Kolmogorov-Smirnov
            ks_stat, p_val = stats.kstest(data, dist.name, args=params)
            
            pdf_y = dist.pdf(x_plot, *params)
            
            # Graficar función de densidad teórica
            fig.add_trace(
                go.Scatter(
                    x=x_plot, 
                    y=pdf_y, 
                    mode="lines", 
                    name=name,
                    showlegend=(idx == 0)
                ),
                row=row, col=col_idx
            )
            
            # Criterio: menor estadística KS -> mejor ajuste
            if ks_stat < best_stat:
                best_stat = ks_stat
                best_dist_name = name
                
        except Exception:
            continue
            
    best_fits_summary[f"{h}h"] = (best_dist_name, best_stat)

# 3. Formatear y desplegar gráfico interactivo
fig.update_layout(
    title_text="Ajuste de Distribuciones por Ventana de Acumulación (Eventos > 0 mm)",
    height=900,
    width=1100,
    template="plotly_white"
)

fig.show()

# 4. Resumen del mejor ajuste por serie
print("=" * 65)
print("RESUMEN DE MEJOR AJUSTE POR SERIE TEMPORAL (Criterio KS-test)")
print("=" * 65)
for h_str, (dist_name, ks_val) in best_fits_summary.items():
    print(f"Ventana {h_str:>3} -> Mejor Función: {dist_name:<12} (Distancia KS: {ks_val:.4f})")
print("=" * 65)